# JEV-V4b — long-context Jev model: Transformer (split session)

V4's combined ~3.5h run doesn't fit Colab's runtime limit (reclaimed twice
mid-finetune), so V4 runs as two single-model sessions: **V4a** trains
jev-v8a separately, **V4b** trains jev-transformer here. Protocol, seeds,
data, and hyperparams are identical to the V4 spec — the two sessions are
directly comparable, and their result files combine into V4's tables.

This session: **jev-transformer** only.
- **taskC** (new): long-context retrieval-decision — a 2k–8k-char passage,
  4 options, exactly one sentence taken verbatim from the passage.
  2000/500/500 train/val/test. Fully automatic, no LLM needed.
- Model: ~15.09M params (DIM 352 × 8 layers × 8 heads; pos_emb sized to 8192
  for long contexts).
- Stage 0: char-LM pre-train, 1500 steps, short 128-token windows (as V3).
- Stage 1: end-to-end decision finetune on taskC only (backbone unfrozen),
  micro-batch 2 with ×4 gradient accumulation, 6 epochs.
- Stage 2: temperature scaling.
- **Latency sweep**: T = 256…8192, batch=1, eager fp32, 10 warmup / 100 timed.
  CUDA OOM is recorded as a finding.


## 0. Setup

In [ ]:
import torch
import triton
print("torch", torch.__version__)
print("triton", triton.__version__)
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
device = "cuda" if torch.cuda.is_available() else "cpu"
assert torch.cuda.is_available(), "JEV-V4 needs the T4 GPU"
torch.manual_seed(0)
import math, random, re, time, json
random.seed(0)


## 1. Data — identical corpus recipe to V1/V2/V3

In [ ]:
import os, urllib.request
os.makedirs("data", exist_ok=True)
URLS = {
    "tinyshakespeare.txt": "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "alice.txt":           "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "frankenstein.txt":    "https://www.gutenberg.org/cache/epub/84/pg84.txt",
    "pride.txt":           "https://www.gutenberg.org/cache/epub/1342/pg1342.txt",
}
parts = []
for name, url in URLS.items():
    p = f"data/{name}"
    if not os.path.exists(p):
        urllib.request.urlretrieve(url, p)
    with open(p, encoding="utf-8", errors="ignore") as f:
        parts.append(f.read())
text = "\n".join(parts)
chars = sorted(set(text)); vocab = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n = int(0.95 * len(data)); train_data, val_data = data[:n], data[n:]
nl_id = stoi.get("\n")
print(f"corpus {len(text)/1e6:.2f} MB, vocab {vocab}, newline_id={nl_id}")
assert nl_id is not None, "newline not in vocab!"

SEQ, BS = 128, 32   # stage-0 pre-training stays on SHORT windows (as V3)
def get_batch(split="train"):
    d = train_data if split == "train" else val_data
    i = torch.randint(0, len(d) - SEQ - 1, (BS,))
    x = torch.stack([d[j:j+SEQ] for j in i]).to(device)
    y = torch.stack([d[j+1:j+SEQ+1] for j in i]).to(device)
    return x, y


## 2. Models — inline models.py + triton_kernels.py (same as V1/V2/V3)

In [ ]:
"""
Linear-recurrent replacement for Transformer self-attention.

Experiment: take a standard decoder-only Transformer, rip out the
causal self-attention layer, and replace it with a *moving linear
equation* that compresses the past into a fixed-size running state:

    h_t = d * h_{t-1} + (1 - d) * (B x_t)        (linear recurrence)
    y_t = C h_t                                   (readout)

where d in (0,1)^S is a learned per-channel decay, B/C are learned
projections, and h has FIXED size S independent of sequence length.

This is O(T) time and O(1) memory w.r.t. context length, vs O(T^2)
for attention. The question: how much modeling power do we lose?
"""

import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------
# Token mixing layers
# ----------------------------------------------------------------------------

class SelectiveSegmentedState(nn.Module):
    """V5: gated segmented state-space mixer.

    Replaces the static decay with two dynamic mechanisms:

    1. Dynamic Selective Gate (the model's "will"):
         g_t = sigmoid(W_g x_t + b_g)      in (0,1), per token & channel
       acts as a learned per-token filter: low-information tokens get
       g_t -> 0 (forgotten fast), salient tokens get g_t -> 1 (kept).

    2. Hard Memory Reset Mask (segment boundaries):
         r_t = 1 if token t is a structural boundary (e.g. newline)
         h_t = (1 - r_t) * (g_t * h_{t-1} + (1 - g_t) * u_t)
       when r_t = 1 the state is multiplied by zero: the past memory
       buffer is completely destroyed and the new segment starts fresh.
       (The residual stream still carries x_t itself forward.)

    The recurrence stays affine in h (h_t = a_t*h_{t-1} + b_t with
    a_t = (1-r_t)*g_t, b_t = (1-r_t)*(1-g_t)*u_t), so the parallel
    associative scan still applies: O(T) work, log2(T) parallel passes.
    State dim stays frozen at `state_dim` (VRAM edge preserved).
    """
    def __init__(self, dim, state_dim=256, dropout=0.0):
        super().__init__()
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)  # bias = per-channel prior
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        # x: (B,T,dim); reset: (B,T) binary, 1 = boundary token
        u = self.in_proj(x)                    # (B,T,S)
        g = torch.sigmoid(self.gate_proj(x))   # (B,T,S) dynamic gate
        if reset is not None:
            keep = (1.0 - reset).unsqueeze(-1)  # (B,T,1)
            a = keep * g
            b = keep * (1.0 - g) * u
        else:
            a = g
            b = (1.0 - g) * u
        h_seq = parallel_scan_ab(a, b)          # (B,T,S)
        return self.dropout(self.out_proj(h_seq))


class CausalSelfAttention(nn.Module):
    """Standard multi-head causal self-attention (the baseline)."""
    def __init__(self, dim, n_heads, dropout=0.0):
        super().__init__()
        assert dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.out = nn.Linear(dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)                       # (B,T,H,D)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))  # (B,H,T,D)
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=self.dropout.p if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(y)


def parallel_scan_ab(A, B):
    """Parallel associative scan (Hillis-Steele) for h_t = A_t*h_{t-1} + B_t.

    A, B: (B, T, S) per-step coefficients. Returns H: (B, T, S).
    log2(T) fully-parallel passes instead of T sequential steps.
    """
    Bsz, T, S = A.shape
    A = A.contiguous()
    Bh = B.contiguous()
    idx = torch.arange(T, device=A.device).view(1, T, 1)
    o = 1
    while o < T:
        m = idx >= o
        A_sh = torch.roll(A, o, dims=1)
        B_sh = torch.roll(Bh, o, dims=1)
        # compose segment [t-o, t-1] then [t]:
        #   a_new = a_t * a_{t-o},  b_new = a_t * b_{t-o} + b_t
        A_new = A * A_sh
        B_new = A * B_sh + Bh
        A = torch.where(m, A_new, A)
        Bh = torch.where(m, B_new, Bh)
        o *= 2
    return Bh


def parallel_linear_recurrence(u, decay):
    """Parallel associative scan (Hillis-Steele) for h_t = d*h_{t-1} + c_t.

    Reformulates the sequential loop as a tree reduction:
      - each step is an affine map  h -> a*h + b, packed as pair (a, b)
      - pair composition is associative:
            (a1,b1) then (a2,b2)  =  (a2*a1, a2*b1 + b2)
      - a parallel prefix over pairs yields every h_t at once

    log2(T) fully-parallel passes instead of T sequential steps.
    u: (B, T, S) inputs already projected; decay: (S,) in (0, 1).
    Returns h: (B, T, S) — identical math to the for-loop version.
    """
    Bsz, T, S = u.shape
    c = (1.0 - decay) * u
    A = decay.expand(Bsz, T, S).contiguous()   # a-coefficients
    return parallel_scan_ab(A, c)


class LinearRunningState(nn.Module):
    """Attention replacement: fixed-size running state, linear recurrence.

    h_t = decay * h_{t-1} + (1 - decay) * (B x_t)
    y_t = C h_t

    - decay: learned per-state-channel forget factor in (0, 1)
    - B:     input  projection  dim -> state_dim
    - C:     output projection  state_dim -> dim
    State size is FIXED (does not grow with context length).
    mode="loop": native Python for-loop (sequential, slow on GPU)
    mode="scan": parallel associative scan (log2(T) parallel passes)
    """
    def __init__(self, dim, state_dim=None, dropout=0.0, mode="loop"):
        super().__init__()
        self.mode = mode
        self.state_dim = state_dim or dim
        # init decay ~ U(0.9, 0.999) -> long-ish memory at start
        init = torch.empty(self.state_dim).uniform_(math.log(0.9 / 0.1),
                                                   math.log(0.999 / 0.001))
        self.logit_decay = nn.Parameter(init)
        self.in_proj = nn.Linear(dim, self.state_dim, bias=False)
        self.out_proj = nn.Linear(self.state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        B, T, _ = x.shape
        decay = torch.sigmoid(self.logit_decay)          # (S,) in (0,1)
        u = self.in_proj(x)                              # (B,T,S)
        if self.mode == "scan":
            h_seq = parallel_linear_recurrence(u, decay)  # (B,T,S)
        else:
            keep = 1.0 - decay
            h = torch.zeros(B, self.state_dim, device=x.device, dtype=x.dtype)
            outs = []
            for t in range(T):
                h = decay * h + keep * u[:, t]
                outs.append(h)
            h_seq = torch.stack(outs, dim=1)              # (B,T,S)
        return self.dropout(self.out_proj(h_seq))


# ----------------------------------------------------------------------------
# Blocks & models (identical except for the mixing layer)
# ----------------------------------------------------------------------------

class Block(nn.Module):
    def __init__(self, dim, n_heads, mixer, dropout=0.0, ffn_hidden=None):
        super().__init__()
        ffn_hidden = ffn_hidden or 4 * dim
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.mix = mixer
        self.mlp = nn.Sequential(
            nn.Linear(dim, ffn_hidden), nn.GELU(),
            nn.Linear(ffn_hidden, dim), nn.Dropout(dropout))

    def forward(self, x, reset=None):
        x = x + self.mix(self.ln1(x), reset)
        x = x + self.mlp(self.ln2(x))
        return x


class TinyLM(nn.Module):
    """Decoder-only LM. mixer_fn(dim, n_heads) builds the token-mixing layer."""
    def __init__(self, vocab, dim=128, n_layers=4, n_heads=4,
                 seq_len=128, mixer_fn=None, dropout=0.0, ffn_hidden=None,
                 newline_id=None):
        super().__init__()
        self.seq_len = seq_len
        self.newline_id = newline_id  # boundary token id for reset mask
        self.tok_emb = nn.Embedding(vocab, dim)
        self.pos_emb = nn.Embedding(seq_len, dim)
        self.blocks = nn.ModuleList(
            [Block(dim, n_heads, mixer_fn(dim, n_heads), dropout, ffn_hidden)
             for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(
            torch.arange(T, device=idx.device))[None]
        reset = None
        if self.newline_id is not None:
            reset = (idx == self.newline_id).float()  # (B,T) boundary mask
        for blk in self.blocks:
            x = blk(x, reset)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new, temp=1.0, top_k=None):
        for _ in range(max_new):
            ctx = idx[:, -self.seq_len:]
            logits, _ = self(ctx)
            logits = logits[:, -1, :] / max(temp, 1e-6)
            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx


def count_params(m):
    return sum(p.numel() for p in m.parameters())


def build_models(vocab, dim=128, n_layers=4, n_heads=4, seq_len=128,
                 state_dim=None, dropout=0.0):
    attn = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                  mixer_fn=lambda d, h: CausalSelfAttention(d, h, dropout),
                  dropout=dropout)
    lin = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                 mixer_fn=lambda d, h: LinearRunningState(d, state_dim, dropout),
                 dropout=dropout)
    return attn, lin


"""(inlined) V6/V7/V8-A: Triton-fused scans.

Fuses the V5 recurrence (per the V6 execution plan):
  Step A: the 256-wide hidden state lives in SRAM registers for the whole
          sequence -- loaded once per batch element, never spilled to HBM.
  Step B: sigmoid gate, hard reset mask, and the linear update
          h = keep*g*h + keep*(1-g)*u run in registers, every timestep.
  Step C: only h_t (plus a_t, saved for the backward pass) hit HBM.
          The parallel scan's O(log T) intermediate (B,T,S) tensors vanish.

Math is IDENTICAL to V5's SelectiveSegmentedState -- only the execution
path changes. Zero new parameters.
"""
import triton
import triton.language as tl

BLOCK_S = 256  # state dim; the whole state fits in SRAM


@triton.jit
def _sel_scan_fwd_kernel(u_ptr, gp_ptr, keep_ptr, h_ptr, a_ptr,
                         T, BLOCK: tl.constexpr):
    """Forward: h_t = a_t*h_{t-1} + b_t with a_t = keep_t*g_t,
    b_t = keep_t*(1-g_t)*u_t, g_t = sigmoid(gp_t)."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    u_b = u_ptr + b * bs
    gp_b = gp_ptr + b * bs
    h_b = h_ptr + b * bs
    a_b = a_ptr + b * bs
    k_b = keep_ptr + b * T
    h = tl.zeros([BLOCK], dtype=tl.float32)          # Step A: state in SRAM
    for t in range(T):
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        gp_t = tl.load(gp_b + t * BLOCK + offs).to(tl.float32)
        k = tl.load(k_b + t).to(tl.float32)
        g = 1.0 / (1.0 + tl.exp(-gp_t))              # Step B: gate in registers
        at = k * g
        h = at * h + k * (1.0 - g) * u_t             # Step B: update in registers
        tl.store(h_b + t * BLOCK + offs, h)          # Step C: outputs to HBM
        tl.store(a_b + t * BLOCK + offs, at)


@triton.jit
def _sel_scan_bwd_kernel(dh_ptr, hs_ptr, ap_ptr, u_ptr, gp_ptr, keep_ptr,
                         du_ptr, dgp_ptr, T, BLOCK: tl.constexpr):
    """Backward. hs is (B,T+1,S) with hs[t] = h_{t-1} (hs[0] = 0);
    ap is (B,T+1,S) with ap[t+1] = a_{t+1} (ap[T] = 0).
    D_t = gout_t + a_{t+1}*D_{t+1}; du_t = k(1-g)D_t;
    dgp_t = k*g*(1-g)*(h_{t-1}-u_t)*D_t."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    bsp = (T + 1) * BLOCK
    dh_b = dh_ptr + b * bs
    hs_b = hs_ptr + b * bsp
    ap_b = ap_ptr + b * bsp
    u_b = u_ptr + b * bs
    gp_b = gp_ptr + b * bs
    k_b = keep_ptr + b * T
    du_b = du_ptr + b * bs
    dgp_b = dgp_ptr + b * bs
    dnext = tl.zeros([BLOCK], dtype=tl.float32)      # D_{t+1}, SRAM-resident
    for ti in range(T):
        t = T - 1 - ti
        gout = tl.load(dh_b + t * BLOCK + offs).to(tl.float32)
        a_next = tl.load(ap_b + (t + 1) * BLOCK + offs).to(tl.float32)
        D = gout + a_next * dnext
        h_prev = tl.load(hs_b + t * BLOCK + offs).to(tl.float32)
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        gp_t = tl.load(gp_b + t * BLOCK + offs).to(tl.float32)
        k = tl.load(k_b + t).to(tl.float32)
        g = 1.0 / (1.0 + tl.exp(-gp_t))
        omg = 1.0 - g
        tl.store(du_b + t * BLOCK + offs, k * omg * D)
        tl.store(dgp_b + t * BLOCK + offs, k * g * omg * (h_prev - u_t) * D)
        dnext = D


class _SelScanFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, u, gp, keep):
        assert u.is_cuda and u.dtype == torch.float32, "V6 kernel: CUDA fp32"
        assert u.shape == gp.shape and u.dim() == 3
        assert keep.shape == u.shape[:2]
        assert u.shape[2] == BLOCK_S
        B, T, S = u.shape
        u, gp, keep = u.contiguous(), gp.contiguous(), keep.contiguous()
        h = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        a = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        _sel_scan_fwd_kernel[(B,)](u, gp, keep, h, a, T,
                                   BLOCK=BLOCK_S, num_warps=4)
        ctx.save_for_backward(u, gp, keep, h, a)
        return h

    @staticmethod
    def backward(ctx, dh):
        u, gp, keep, h, a = ctx.saved_tensors
        B, T, S = u.shape
        dh = dh.contiguous()
        hs = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        hs[:, 1:] = h    # hs[t] = h_{t-1}
        ap = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        ap[:, :T] = a    # ap[t+1] = a_{t+1}
        du = torch.empty_like(u)
        dgp = torch.empty_like(gp)
        _sel_scan_bwd_kernel[(B,)](dh, hs, ap, u, gp, keep, du, dgp, T,
                                   BLOCK=BLOCK_S, num_warps=4)
        return du, dgp, None


def triton_selective_scan(u, gp, keep):
    """Fused selective scan. u, gp: (B,T,256) fp32 CUDA; keep: (B,T) fp32.
    Returns h: (B,T,256). Differentiable."""
    return _SelScanFn.apply(u, gp, keep)


class SelectiveSegmentedStateTriton(nn.Module):
    """V6 mixer: IDENTICAL math and parameters to V5's SelectiveSegmentedState.

    The only change is execution: the recurrence runs in the fused Triton
    kernel (Step A/B/C) instead of the PyTorch Hillis-Steele scan.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V6 kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        gp = self.gate_proj(x).float().contiguous()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_selective_scan(u, gp, keep)
        return self.dropout(self.out_proj(h_seq.to(x.dtype)))


class SelectiveSegmentedStateV7(nn.Module):
    """V7 mixer: V6's Triton-fused selective scan + two literature grafts.

    Graft #1 — output gate + state normalization (HGRN2 / Mamba-2 / GDN):
        y_t = C · (o_t ⊙ RMSNorm(h_t)),   o_t = σ(W_o x_t + b_o)
    States accumulate over segments so readout scale drifts; every strong
    post-Mamba block gates the readout. RMSNorm here is parameter-free
    (no scale vector) to keep the parameter budget exact.

    Graft #3 — layer-graded forget floor (HGRN, NeurIPS'23):
        g_t = σ(W_g x_t + b_g + floor_l)
    floor_l is a per-layer learnable scalar, initialized monotonically
    increasing bottom→top, so low layers forget fast (spelling-level) and
    top layers retain (phrase-level). Complements the newline hard reset.

    The recurrence math is UNCHANGED → the same fused Triton kernel as V6.
    With FFN 1023 the budget lands EXACTLY on the attention baseline.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0, forget_floor_init=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V7 kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)
        self.out_gate_proj = nn.Linear(dim, state_dim)
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.forget_floor = nn.Parameter(
            torch.tensor(float(forget_floor_init)))
        self.dropout = nn.Dropout(dropout)
        self.eps = 1e-6

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        gp = (self.gate_proj(x) + self.forget_floor).float().contiguous()
        og = self.out_gate_proj(x).float()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_selective_scan(u, gp, keep)          # (B,T,S)
        h_norm = h_seq / torch.sqrt(
            (h_seq ** 2).mean(dim=-1, keepdim=True) + self.eps)
        y = self.out_proj((torch.sigmoid(og) * h_norm).to(x.dtype))
        return self.dropout(y)


# ----------------------------------------------------------------------------
# V8-A: diagonal gated delta-rule scan
# ----------------------------------------------------------------------------

@triton.jit
def _delta_scan_fwd_kernel(u_ptr, ap_ptr, bp_ptr, kp_ptr, keep_ptr,
                           h_ptr, a_ptr, T, BLOCK: tl.constexpr):
    """Forward: h_t = a_t*h_{t-1} + b_t with
        alpha = sigmoid(ap_t), beta = sigmoid(bp_t), kk = sigmoid(kp_t),
        v = silu(u_t),
        a_t = keep_t * alpha * (1 - beta*kk^2),
        b_t = keep_t * beta * kk * v.
    Diagonal restriction of Gated DeltaNet's S_t = a_t S_{t-1}(I-b_t k k^T)
    + b_t v k^T to per-channel 1x1 systems. Strictly contractive:
    sigmoid bounds keep (1-beta*kk^2) in (0,1)."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    u_b = u_ptr + b * bs
    ap_b = ap_ptr + b * bs
    bp_b = bp_ptr + b * bs
    kp_b = kp_ptr + b * bs
    h_b = h_ptr + b * bs
    a_b = a_ptr + b * bs
    k_b = keep_ptr + b * T
    h = tl.zeros([BLOCK], dtype=tl.float32)          # state in SRAM
    for t in range(T):
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        ap_t = tl.load(ap_b + t * BLOCK + offs).to(tl.float32)
        bp_t = tl.load(bp_b + t * BLOCK + offs).to(tl.float32)
        kp_t = tl.load(kp_b + t * BLOCK + offs).to(tl.float32)
        kr = tl.load(k_b + t).to(tl.float32)
        alpha = 1.0 / (1.0 + tl.exp(-ap_t))
        beta = 1.0 / (1.0 + tl.exp(-bp_t))
        kk = 1.0 / (1.0 + tl.exp(-kp_t))
        su = 1.0 / (1.0 + tl.exp(-u_t))
        v = u_t * su                                 # SiLU in registers
        at = kr * alpha * (1.0 - beta * kk * kk)
        bt = kr * beta * kk * v
        h = at * h + bt
        tl.store(h_b + t * BLOCK + offs, h)
        tl.store(a_b + t * BLOCK + offs, at)


@triton.jit
def _delta_scan_bwd_kernel(dh_ptr, hs_ptr, apn_ptr, u_ptr, ap_ptr, bp_ptr,
                           kp_ptr, keep_ptr, du_ptr, dap_ptr, dbp_ptr,
                           dkp_ptr, T, BLOCK: tl.constexpr):
    """Backward. hs is (B,T+1,S) with hs[t] = h_{t-1} (hs[0] = 0);
    apn is (B,T+1,S) with apn[t+1] = a_{t+1} (apn[T] = 0).
    Adjoint: D_t = gout_t + a_{t+1}*D_{t+1}; then per-step
      du  = D*kr*beta*kk*silu'(u)
      dap = D*h_prev*kr*alpha*(1-alpha)*(1-beta*kk^2)
      dbp = D*kr*beta*(1-beta)*(kk*v - alpha*kk^2*h_prev)
      dkp = D*kr*beta*kk*(1-kk)*(v - 2*alpha*kk*h_prev)
    (hand-derived; verified against autograd on CPU before transcription)."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    bsp = (T + 1) * BLOCK
    dh_b = dh_ptr + b * bs
    hs_b = hs_ptr + b * bsp
    apn_b = apn_ptr + b * bsp
    u_b = u_ptr + b * bs
    ap_b = ap_ptr + b * bs
    bp_b = bp_ptr + b * bs
    kp_b = kp_ptr + b * bs
    k_b = keep_ptr + b * T
    du_b = du_ptr + b * bs
    dap_b = dap_ptr + b * bs
    dbp_b = dbp_ptr + b * bs
    dkp_b = dkp_ptr + b * bs
    dnext = tl.zeros([BLOCK], dtype=tl.float32)      # D_{t+1}, SRAM-resident
    for ti in range(T):
        t = T - 1 - ti
        gout = tl.load(dh_b + t * BLOCK + offs).to(tl.float32)
        a_next = tl.load(apn_b + (t + 1) * BLOCK + offs).to(tl.float32)
        D = gout + a_next * dnext
        h_prev = tl.load(hs_b + t * BLOCK + offs).to(tl.float32)
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        ap_t = tl.load(ap_b + t * BLOCK + offs).to(tl.float32)
        bp_t = tl.load(bp_b + t * BLOCK + offs).to(tl.float32)
        kp_t = tl.load(kp_b + t * BLOCK + offs).to(tl.float32)
        kr = tl.load(k_b + t).to(tl.float32)
        alpha = 1.0 / (1.0 + tl.exp(-ap_t))
        beta = 1.0 / (1.0 + tl.exp(-bp_t))
        kk = 1.0 / (1.0 + tl.exp(-kp_t))
        su = 1.0 / (1.0 + tl.exp(-u_t))
        v = u_t * su
        silu_p = su * (1.0 + u_t * (1.0 - su))
        k2 = kk * kk
        omb = 1.0 - beta * k2
        tl.store(du_b + t * BLOCK + offs,
                 D * kr * beta * kk * silu_p)
        tl.store(dap_b + t * BLOCK + offs,
                 D * h_prev * kr * alpha * (1.0 - alpha) * omb)
        tl.store(dbp_b + t * BLOCK + offs,
                 D * kr * beta * (1.0 - beta) * (kk * v - alpha * k2 * h_prev))
        tl.store(dkp_b + t * BLOCK + offs,
                 D * kr * beta * kk * (1.0 - kk) * (v - 2.0 * alpha * kk * h_prev))
        dnext = D


def _delta_scan_cpu(u, ap, bp, kp, keep):
    """CPU reference for the delta scan: differentiable pure-PyTorch loop.

    Used ONLY for local (CPU) verification. On CUDA the fused Triton
    kernel above is used instead (gate-checked against this math on Colab).
    """
    B, T, S = u.shape
    kr = keep.unsqueeze(-1)
    alpha = torch.sigmoid(ap)
    beta = torch.sigmoid(bp)
    kk = torch.sigmoid(kp)
    v = u * torch.sigmoid(u)                          # SiLU
    h = torch.zeros(B, S, dtype=u.dtype, device=u.device)
    hs = []
    for t in range(T):
        a = kr[:, t] * alpha[:, t] * (1.0 - beta[:, t] * kk[:, t] ** 2)
        bb = kr[:, t] * beta[:, t] * kk[:, t] * v[:, t]
        h = a * h + bb
        hs.append(h)
    return torch.stack(hs, dim=1)


class _DeltaScanFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, u, ap, bp, kp, keep):
        assert u.is_cuda and u.dtype == torch.float32, \
            "V8-A kernel: CUDA fp32"
        for t in (u, ap, bp, kp):
            assert t.shape == u.shape and t.dim() == 3
        assert keep.shape == u.shape[:2]
        assert u.shape[2] == BLOCK_S
        B, T, S = u.shape
        u, ap, bp, kp = u.contiguous(), ap.contiguous(), bp.contiguous(), kp.contiguous()
        keep = keep.contiguous()
        h = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        a = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        _delta_scan_fwd_kernel[(B,)](u, ap, bp, kp, keep, h, a, T,
                                    BLOCK=BLOCK_S, num_warps=4)
        ctx.save_for_backward(u, ap, bp, kp, keep, h, a)
        return h

    @staticmethod
    def backward(ctx, dh):
        u, ap, bp, kp, keep, h, a = ctx.saved_tensors
        B, T, S = u.shape
        dh = dh.contiguous()
        hs = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        hs[:, 1:] = h      # hs[t] = h_{t-1}
        apn = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        apn[:, :T] = a     # apn[t+1] = a_{t+1}
        du = torch.empty_like(u)
        dap = torch.empty_like(ap)
        dbp = torch.empty_like(bp)
        dkp = torch.empty_like(kp)
        _delta_scan_bwd_kernel[(B,)](dh, hs, apn, u, ap, bp, kp, keep,
                                    du, dap, dbp, dkp, T,
                                    BLOCK=BLOCK_S, num_warps=4)
        return du, dap, dbp, dkp, None


def triton_delta_scan(u, ap, bp, kp, keep):
    """Fused diagonal delta-rule scan.

    u: raw Bx (SiLU applied in-kernel); ap/bp/kp: pre-sigmoid
    alpha/beta/key projections; keep: (B,T) reset mask.
    (B,T,256) fp32; CUDA -> Triton kernel, CPU -> differentiable
    reference loop (verification only). Returns h: (B,T,256).
    """
    if u.is_cuda:
        return _DeltaScanFn.apply(u, ap, bp, kp, keep)
    return _delta_scan_cpu(u, ap, bp, kp, keep)


class SelectiveSegmentedStateV8A(nn.Module):
    """V8-A mixer: diagonal gated delta rule on V7's chassis.

    Per channel (delta rule = erase-before-write, Gated DeltaNet Eq.10
    restricted to 1x1 systems):
        alpha_t = sigmoid(W_a x_t + b_a + floor_l)   # reuse gate_proj+floor
        beta_t  = sigmoid(W_b x_t + b_b),  b_b = 0   # NEW write_proj
        k_t     = sigmoid(W_k x_t + b_k),  b_k = +2  # NEW key_proj
        v_t     = SiLU(B x_t)                        # reuse in_proj + SiLU
        a_t = alpha_t * (1 - beta_t * k_t^2)
        b_t = beta_t * k_t * v_t
        h_t = (1 - r_t) * (a_t .* h_{t-1} + b_t)
        y_t = C . (o_t .* RMSNorm(h_t)), o_t = sigmoid(W_o x_t)

    Init puts the model near V7's convex-blend regime (k~=0.88, beta~=0.5)
    and the sigmoid bounds keep every step strictly contractive.
    With FFN 766 the budget lands within the relaxed <=3000 gate.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0, forget_floor_init=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V8-A kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)   # B
        self.gate_proj = nn.Linear(dim, state_dim)             # W_alpha
        self.write_proj = nn.Linear(dim, state_dim)            # W_beta (NEW)
        self.key_proj = nn.Linear(dim, state_dim)              # W_k (NEW)
        self.out_gate_proj = nn.Linear(dim, state_dim)         # W_o
        self.out_proj = nn.Linear(state_dim, dim, bias=False)  # C
        self.forget_floor = nn.Parameter(
            torch.tensor(float(forget_floor_init)))
        self.write_proj.bias.data.zero_()       # b_beta = 0  -> beta ~= 0.5
        self.key_proj.bias.data.fill_(2.0)       # b_k = +2.0  -> k ~= 0.88
        self.dropout = nn.Dropout(dropout)
        self.eps = 1e-6
        self.last_state = None  # stashed h_seq for stability logging (no grad)

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        ap = (self.gate_proj(x) + self.forget_floor).float().contiguous()
        bp = self.write_proj(x).float().contiguous()
        kp = self.key_proj(x).float().contiguous()
        og = self.out_gate_proj(x).float()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_delta_scan(u, ap, bp, kp, keep)      # (B,T,S)
        self.last_state = h_seq.detach()  # stability logging only; no numerics change
        h_norm = h_seq / torch.sqrt(
            (h_seq ** 2).mean(dim=-1, keepdim=True) + self.eps)
        y = self.out_proj((torch.sigmoid(og) * h_norm).to(x.dtype))
        return self.dropout(y)


## 3. Build backbone (V4b) — ~15M params, long-context positions, budget gate

In [ ]:
# V4b: transformer backbone ONLY (jev-v8a trains in the V4a session).
# Same construction recipe as V4: DIM 352 x 8 layers x 8 heads, pos_emb sized
# to CTX_LONG=8192 (V3's 128-row table would IndexError past T=128).
# Comparability note: torch.manual_seed(0) is set in Setup exactly as in V4,
# but only one backbone is constructed here, so implicit global-RNG draws for
# weight init / stage-0 batches differ from the combined run. All EXPLICIT
# seeds are identical: taskC uses random.Random(0), head init uses
# torch.manual_seed(1234), batch order uses torch.Generator().manual_seed(0).
DIM, LAYERS, HEADS, CTX_LONG = 352, 8, 8, 8192
attention = TinyLM(vocab, DIM, LAYERS, HEADS, CTX_LONG,
    mixer_fn=lambda d, h: CausalSelfAttention(d, h)).to(device)
# JevChoiceHead (defined in section 6): q,k = Linear(DIM,DIM,bias=False)
HEAD_P = 2 * DIM * DIM
pa = sum(p.numel() for p in attention.parameters()) + HEAD_P
print(f"jev-transformer total (backbone+head): {pa/1e6:.3f}M params")
assert 14.5e6 <= pa <= 15.5e6, f"transformer outside 15M band: {pa}"
print("BUDGET GATE PASSED (15M band)")
# Long-context smoke: catches pos_emb indexing issues at T=8192 BEFORE the
# expensive training stages.
with torch.no_grad():
    _probe = torch.randint(0, vocab, (1, CTX_LONG)).to(device)
    _lg, _ = attention(_probe)
    assert _lg.shape == (1, CTX_LONG, vocab), ("attention", _lg.shape)
    print("long-context forward smoke OK (T=8192: pos_emb)")
del _probe
torch.cuda.empty_cache()
MNAME, BB = "jev-transformer", attention


## 4. Stage 0 — LM pre-train backbone (1500 steps, short windows)

In [ ]:
STEPS, EVAL, LR = 1500, 250, 3e-4
opt = torch.optim.AdamW(BB.parameters(), lr=LR)

def val_loss(m):
    m.eval()
    with torch.no_grad():
        return sum(float(m(*get_batch("val"))[1]) for _ in range(10)) / 10

for step in range(1, STEPS + 1):
    BB.train()
    xb, yb = get_batch("train")
    _, loss = BB(xb, yb)
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(BB.parameters(), 1.0)
    opt.step()
    if step % EVAL == 0 or step == 1:
        vl = val_loss(BB)
        print(f"step {step:5d} | {MNAME}: val {vl:.3f} (ppl {math.exp(vl):.1f})",
              flush=True)
print("BACKBONETRAININGDONE")
print(f"FINAL {MNAME}: val {val_loss(BB):.3f}")
# NOTE: backbone stays UNFROZEN — V4 finetunes end-to-end on taskC (like V2/V3)


## 5. taskC — long-context retrieval-decision (2000–8000 char passages)

In [ ]:
# taskC: "which of these 4 sentences appears verbatim in the passage?"
# Passages concatenate corpus sentences to a uniform-random 2000-8000 chars.
# 1 TRUE option = a sentence drawn verbatim from THIS passage; 3 distractors =
# sentences from elsewhere (rejected if they appear in this passage, so every
# example is unambiguous). Fully automatic, no LLM needed.
N_TRAIN, N_VAL, N_TEST = 2000, 500, 500
K_OPTS, OPT_C = 4, 224   # options are full sentences (<=200 chars)
rng = random.Random(0)

sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if 30 <= len(s.strip()) <= 200]
print(f"usable sentences: {len(sents)}")
# Worst case per passage: 8000-char target / 30-char min sentence ~= 267
# distinct sentences. Passages sample WITH replacement across the 3000
# examples (the corpus is ~5MB; 3000 x ~5000-char passages would need ~15MB
# of unique text), but WITHOUT replacement within a passage.
assert len(sents) >= 1000, (
    f"too few distinct sentences for 3000 long passages: have {len(sents)}, "
    f"need >= 1000 (worst case ~267 distinct 30-char sentences per 8000-char passage)")

def make_passage():
    target = rng.randint(2000, 8000)
    idxs = rng.sample(range(len(sents)), 400)
    chosen, total = [], 0
    for i in idxs:
        s = sents[i]
        if total + len(s) + 1 > CTX_LONG:
            continue
        chosen.append(s)
        total += len(s) + 1
        if total >= target:
            break
    passage = " ".join(chosen)
    assert 2000 <= len(passage) <= CTX_LONG, f"bad passage length {len(passage)}"
    return passage, chosen

def make_taskC(n):
    ctx, opts, lab = [], [], []
    for _ in range(n):
        passage, constituents = make_passage()
        const_set = set(constituents)
        true = rng.choice(constituents)
        distract = []
        while len(distract) < K_OPTS - 1:
            d = sents[rng.randrange(len(sents))]
            if d not in const_set and d not in distract and d not in passage:
                distract.append(d)
        options = [true] + distract
        rng.shuffle(options)
        ctx.append(passage); opts.append(options); lab.append(options.index(true))
    return ctx, opts, lab

def tok_pad(s, maxlen):
    ids = [stoi.get(c, 0) for c in s[:maxlen]]
    L = len(ids)
    return ids + [0] * (maxlen - L), L

def to_tensors_c(ctx, opts, lab):
    Bc = len(ctx)
    C = torch.zeros(Bc, CTX_LONG, dtype=torch.long)
    Cl = torch.zeros(Bc, dtype=torch.long)
    O = torch.zeros(Bc, K_OPTS, OPT_C, dtype=torch.long)
    Ol = torch.zeros(Bc, K_OPTS, dtype=torch.long)
    for b in range(Bc):
        ids, L = tok_pad(ctx[b], CTX_LONG); C[b] = torch.tensor(ids); Cl[b] = L
        for k in range(K_OPTS):
            ids, L = tok_pad(opts[b][k], OPT_C); O[b, k] = torch.tensor(ids); Ol[b, k] = L
    return {"ctx": C, "ctx_len": Cl, "opts": O, "opt_len": Ol,
            "label": torch.tensor(lab, dtype=torch.long)}

tasks = {}
for sname, n in [("train", N_TRAIN), ("val", N_VAL), ("test", N_TEST)]:
    tc, to, tl = make_taskC(n)
    tasks[sname] = to_tensors_c(tc, to, tl)
    _lens = tasks[sname]["ctx_len"]
    print(f"taskC {sname}: {n} examples, ctx_len min {_lens.min().item()} "
          f"max {_lens.max().item()} mean {_lens.float().mean().item():.0f}", flush=True)
n_pass = sum(tasks[s]["label"].shape[0] for s in ["train", "val", "test"])
assert n_pass == N_TRAIN + N_VAL + N_TEST == 3000, f"passage count wrong: {n_pass}"
print("random-guess baseline accuracy: 25.0%")


## 6. Jev decision model — JevChoiceHead + end-to-end decision forward

In [ ]:
class JevChoiceHead(torch.nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.q = torch.nn.Linear(dim, dim, bias=False)
        self.k = torch.nn.Linear(dim, dim, bias=False)
        self.scale = dim ** -0.5
    def forward(self, H, Hmask, O):
        Q = self.q(O)                                   # (B,K,D)
        K = self.k(H)                                   # (B,T,D)
        s = torch.einsum('bkd,btd->bkt', Q, K) * self.scale
        s = s.masked_fill(~Hmask[:, None, :], float('-inf'))
        a = torch.softmax(s, dim=-1)                    # (B,K,T)
        C = torch.einsum('bkt,btd->bkd', a, H)          # option-specific ctx
        return torch.einsum('bkd,bkd->bk', C, O)        # (B,K) logits

def bb_forward(model, idx):
    B, T = idx.shape
    x = model.tok_emb(idx) + model.pos_emb(torch.arange(T, device=idx.device))[None]
    reset = (idx == model.newline_id).float() if model.newline_id is not None else None
    for blk in model.blocks:
        x = blk(x, reset)
    return model.ln_f(x)

def encode_opts(model, O, Ol):
    b, K, To = O.shape
    Ho = bb_forward(model, O.reshape(b * K, To))
    ar = torch.arange(To, device=O.device)[None, :]
    om = (ar < Ol.reshape(b * K, 1)).float()
    Ov = (Ho * om[..., None]).sum(1) / Ol.reshape(b * K, 1).clamp_min(1).float()
    return Ov.reshape(b, K, -1)

def decide(model, head, C, Cl, O, Ol):
    H = bb_forward(model, C)
    Ov = encode_opts(model, O, Ol)
    Hmask = torch.arange(H.shape[1], device=H.device)[None, :] < Cl[:, None]
    return head(H, Hmask, Ov)

# shape smoke test on a real long example (no_grad, then release)
_h = JevChoiceHead(DIM).to(device)
_t = tasks["train"]
with torch.no_grad():
    _lg = decide(BB, _h, _t["ctx"][:1].to(device), _t["ctx_len"][:1].to(device),
                 _t["opts"][:1].to(device), _t["opt_len"][:1].to(device))
assert _lg.shape == (1, K_OPTS), f"bad decide shape {_lg.shape}"
print("decide smoke OK (long context):", tuple(_lg.shape),
      f"{sum(p.numel() for p in _h.parameters())} head params")
del _h, _lg
torch.cuda.empty_cache()


## 7. Stage 1 — end-to-end decision finetune on taskC (backbone UNFROZEN)

In [ ]:
# Long contexts (up to 8192 tokens) x 15M params on a 16GB T4:
# micro-batch 2 with x4 gradient accumulation -> effective batch 8.
# 6 epochs x 250 optimizer steps = 1500 steps (V3 used 200 steps at short
# context; 25 epochs here would be ~3h — infeasible, and unnecessary).
import copy
EPOCHS, MICRO, ACCUM, LR_FT = 6, 2, 4, 3e-4
EVAL_B = 8
trained = {}
mname, bb = MNAME, BB
model = copy.deepcopy(bb).to(device)
torch.manual_seed(1234)  # identical head init across both sessions
head = JevChoiceHead(DIM).to(device)
opt = torch.optim.AdamW(list(model.parameters()) + list(head.parameters()),
                        lr=LR_FT)
tr = tasks["train"]; va = tasks["val"]
g = torch.Generator().manual_seed(0)  # identical batch order across sessions
best_acc, best_state = 0.0, None
n_steps = tr["label"].shape[0] // MICRO
assert n_steps % ACCUM == 0, "micro-steps must divide evenly into accum groups"
for ep in range(EPOCHS):
    model.train(); head.train()
    perm = torch.randperm(tr["label"].shape[0], generator=g)
    opt.zero_grad()
    for si in range(n_steps):
        b = perm[si*MICRO:(si+1)*MICRO]
        logits = decide(model, head,
                        tr["ctx"][b].to(device), tr["ctx_len"][b].to(device),
                        tr["opts"][b].to(device), tr["opt_len"][b].to(device))
        loss = F.cross_entropy(logits, tr["label"][b].to(device)) / ACCUM
        loss.backward()
        if (si + 1) % ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(
                list(model.parameters()) + list(head.parameters()), 1.0)
            opt.step()
            opt.zero_grad()
    model.eval(); head.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for i in range(0, va["label"].shape[0], EVAL_B):
            b = slice(i, i + EVAL_B)
            vl = decide(model, head,
                        va["ctx"][b].to(device), va["ctx_len"][b].to(device),
                        va["opts"][b].to(device), va["opt_len"][b].to(device))
            lbl = va["label"][b].to(device)
            correct += (vl.argmax(-1) == lbl).sum().item()
            total += lbl.shape[0]
    acc = correct / total
    print(f"{mname} taskC epoch {ep+1}/{EPOCHS}: val acc {acc*100:.1f}%",
          flush=True)
    if acc > best_acc:
        best_acc = acc
        best_state = (
            {k: v.cpu().clone() for k, v in model.state_dict().items()},
            {k: v.cpu().clone() for k, v in head.state_dict().items()})
ms, hs = best_state
model.load_state_dict({k: v.to(device) for k, v in ms.items()})
head.load_state_dict({k: v.to(device) for k, v in hs.items()})
trained[mname] = (model, head)
print(f"{mname} taskC: best val acc {best_acc*100:.1f}% "
      f"({sum(p.numel() for p in model.parameters()) + sum(p.numel() for p in head.parameters())} params)",
      flush=True)
del ms, hs
torch.cuda.empty_cache()
print("FINETUNEDONE")


## 8. Stage 2 — temperature scaling (calibration) + ECE / Brier

In [ ]:
def fit_temperature(logits, labels):
    T = torch.nn.Parameter(torch.ones(1, device=logits.device))
    opt_t = torch.optim.LBFGS([T], lr=0.1, max_iter=50)
    def closure():
        opt_t.zero_grad()
        loss = F.cross_entropy(logits / T.clamp_min(1e-3), labels)
        loss.backward()
        return loss
    opt_t.step(closure)
    return float(T.clamp_min(1e-3))

def ece_score(probs, labels, n_bins=15):
    conf, pred = probs.max(-1)
    bins = torch.linspace(0, 1, n_bins + 1, device=probs.device)
    e = 0.0
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i + 1])
        if m.sum() > 0:
            acc_b = (pred[m] == labels[m]).float().mean()
            con_b = conf[m].mean()
            e += (m.float().mean() * (acc_b - con_b).abs()).item()
    return e

def brier_score(probs, labels, K=K_OPTS):
    oh = F.one_hot(labels, K).float()
    return float(((probs - oh) ** 2).sum(-1).mean())

def forward_all(model, head, split, batch=EVAL_B):
    t = tasks[split]
    outs = []
    model.eval(); head.eval()
    with torch.no_grad():
        for i in range(0, t["label"].shape[0], batch):
            b = slice(i, i + batch)
            outs.append(decide(model, head,
                               t["ctx"][b].to(device), t["ctx_len"][b].to(device),
                               t["opts"][b].to(device), t["opt_len"][b].to(device)))
    return torch.cat(outs)

results = {}
for mname, (model, head) in trained.items():
    test_logits = forward_all(model, head, "test")
    val_logits = forward_all(model, head, "val")
    te, va = tasks["test"], tasks["val"]
    T = fit_temperature(val_logits, va["label"].to(device))
    raw_p = torch.softmax(test_logits, -1)
    cal_p = torch.softmax(test_logits / T, -1)
    acc = float((test_logits.argmax(-1) == te["label"].to(device)).float().mean())
    results[f"{mname}/taskC"] = {
        "acc": acc, "ece_raw": ece_score(raw_p, te["label"].to(device)),
        "ece_cal": ece_score(cal_p, te["label"].to(device)),
        "brier": brier_score(cal_p, te["label"].to(device)), "temp": T}
    r = results[f"{mname}/taskC"]
    print(f"{mname} taskC: acc {r['acc']*100:.1f}% | "
          f"ECE {r['ece_raw']:.3f} -> {r['ece_cal']:.3f} (T={r['temp']:.2f}) | "
          f"Brier {r['brier']:.3f}", flush=True)
print("CALIBRATIONDONE")


## 9. Latency sweep — O(T) vs O(T^2), T=256..8192

In [ ]:
# The core V4 question: where does attention's latency curve bend?
# Inference-only, batch=1, torch.no_grad, eager fp32 — the honest comparable
# mode (V3 showed compile+fp16 backfires for both at batch=1 tiny scale).
# Each measurement: 10 warmup + 100 timed, cuda.synchronize.
# A CUDA OOM is caught and recorded as "OOM" — an OOM IS a finding
# (attention may OOM where the linear substrate doesn't).
SWEEP_T = [256, 512, 1024, 2048, 4096, 8192]

def sweep_batch(T):
    passage = text[:T]
    assert len(passage) == T, f"passage length {len(passage)} != {T}"
    ids = [stoi.get(c, 0) for c in passage]
    C = torch.tensor([ids], dtype=torch.long).to(device)
    Cl = torch.tensor([T], dtype=torch.long).to(device)
    O = torch.zeros(1, K_OPTS, OPT_C, dtype=torch.long).to(device)
    Ol = torch.zeros(1, K_OPTS, dtype=torch.long).to(device)
    for k in range(K_OPTS):
        oids = [stoi.get(c, 0) for c in sents[k][:OPT_C]]
        O[0, k, :len(oids)] = torch.tensor(oids, device=device)
        Ol[0, k] = len(oids)
    return C, Cl, O, Ol

def bench_T(model, head, T, iters=100):
    model.eval(); head.eval()
    C, Cl, O, Ol = sweep_batch(T)
    try:
        with torch.no_grad():
            for _ in range(10):
                decide(model, head, C, Cl, O, Ol)
            torch.cuda.synchronize(); t0 = time.time()
            for _ in range(iters):
                decide(model, head, C, Cl, O, Ol)
            torch.cuda.synchronize()
        return (time.time() - t0) / iters * 1000
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache()
            return None
        raise

sweep = {}
model, head = trained[MNAME]
for T in SWEEP_T:
    ms = bench_T(model, head, T)
    sweep[(MNAME, T)] = ms
    print(f"{MNAME} T={T:5d}: {'OOM' if ms is None else f'{ms:7.1f} ms/decision'}",
          flush=True)
torch.cuda.empty_cache()

print()
print(f"=== JEV-V4b LATENCY SWEEP ({MNAME}) ===")
print(f"{'T (chars)':>10s} {(MNAME + ' ms'):>18s}")
for T in SWEEP_T:
    a = sweep[(MNAME, T)]
    sa = "OOM" if a is None else f"{a:.1f}"
    print(f"{T:>10d} {sa:>18s}")

# Crossover needs BOTH models' sweeps; the two sessions' result files are
# combined locally into V4's crossover table. Recorded here as pending.
crossover_line = ("crossover: single-model session — combine the V4a and V4b "
                  "sweeps to compute the crossover")
print(crossover_line)
print("LATENCYDONE")


## 10. Save results

In [ ]:
print()
print(f"=== JEV-V4b TEST RESULTS ({MNAME}) ===")
print(f"{'config':22s} {'task':6s} {'acc':>6s} {'ece_raw':>8s} {'ece_cal':>8s} {'brier':>7s}")
r = results[f"{MNAME}/taskC"]
print(f"{MNAME:22s} {'taskC':6s} {r['acc']*100:5.1f}% {r['ece_raw']:8.3f} {r['ece_cal']:8.3f} {r['brier']:7.3f}")
print()
print(f"=== JEV-V4b LATENCY SWEEP ({MNAME}) ===")
print(f"{'T (chars)':>10s} {(MNAME + ' ms'):>18s}")
for T in SWEEP_T:
    a = sweep[(MNAME, T)]
    sa = "OOM" if a is None else f"{a:.1f}"
    print(f"{T:>10d} {sa:>18s}")
print(crossover_line)

sweep_json = {str(T): {MNAME: sweep[(MNAME, T)]} for T in SWEEP_T}
results["latency_sweep"] = sweep_json
results["crossover"] = crossover_line

with open("results_jev_v4b_transformer.json", "w") as f:
    json.dump(results, f, indent=1)
with open("results_jev_v4b_transformer.txt", "w") as f:
    f.write("JEV-V4b RESULTS (test, eager fp32)\n")
    f.write(f"{MNAME}/taskC: " + json.dumps(results[f"{MNAME}/taskC"]) + "\n")
    f.write("latency_sweep: " + json.dumps(sweep_json) + "\n")
    f.write("crossover: " + crossover_line + "\n")
print()
print("SAVE DONE")
